In [ ]:
pip install pandas scipy matplotlib seaborn statsmodels jupyter


In [ ]:
import pandas as pd

# IMPORTING DATASET

In [ ]:
df = pd.read_csv("F:\Portfolio Projects\marketing_AB.csv\marketing_AB.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

Checking Data Quality

In [ ]:
print("MISSING VALUES")
print(df.isnull().sum())

# No missing values found

In [ ]:
print("DUPLICATE ROWS")
print(f"Number of duplicates: {df.duplicated().sum()}")

# No duplicate values 

In [ ]:
df.describe()

In [ ]:
print("UNIQUE VALUES IN KEY COLUMNS")
print(f"Test groups: {df['test group'].unique()}")
print(f"Converted values: {df['converted'].unique()}")

# No discrepancy found

Checking Group-wise Distribution 

In [ ]:
print("GROUP DISTRIBUTION")
group_counts = df.groupby('test group')['user id'].nunique()
print(group_counts)

print("\nPERCENTAGE SPLIT")
total_users = group_counts.sum()
for group, count in group_counts.items():
    print(f"{group}: {count:,} users ({count/total_users*100:.1f}%)")

print(f"\nTotal unique users: {total_users:,}")

In [ ]:
# Calculate conversion rate for each group

conversion_rates = df.groupby('test group')['converted'].agg(['sum', 'count', 'mean'])
conversion_rates.columns = ['Conversions', 'Total Users', 'Conversion Rate']
conversion_rates['Conversion Rate %'] = (conversion_rates['Conversion Rate'] * 100).round(2)

print("CONVERSION RATE BY GROUP")
print(conversion_rates)

Hypothesis Testing

In [ ]:
from scipy import stats
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# Extract values for each group
ad_users = conversion_rates.loc['ad', 'Total Users']
ad_conversions = conversion_rates.loc['ad', 'Conversions']

psa_users = conversion_rates.loc['psa', 'Total Users']
psa_conversions = conversion_rates.loc['psa', 'Conversions']

# Conversion rates
ad_rate = ad_conversions / ad_users
psa_rate = psa_conversions / psa_users

# Run two-proportion z-test
count = np.array([ad_conversions, psa_conversions])
nobs = np.array([ad_users, psa_users])

z_stat, p_value = proportions_ztest(count, nobs, alternative='larger')

print(" HYPOTHESIS TEST RESULTS ")
print(f"Ad Conversion Rate:  {ad_rate:.4f} ({ad_rate*100:.2f}%)")
print(f"PSA Conversion Rate: {psa_rate:.4f} ({psa_rate*100:.2f}%)")
print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"\nSignificance level: 0.05")

if p_value < 0.05:
    print("Result: REJECT the null hypothesis")
    print("The ads perform significantly better than the PSA ✓")
else:
    print("Result: FAIL TO REJECT the null hypothesis")
    print("No significant difference found between ads and PSA")

Important Findings 

In [ ]:
print(" BUSINESS IMPACT ")

# Relative uplift
uplift = (ad_rate - psa_rate) / psa_rate * 100
print(f"Relative uplift from ads: {uplift:.1f}%")

# If PSA group had seen ads instead - how many extra conversions?
extra_conversions = int(psa_users * (ad_rate - psa_rate))
print(f"Extra conversions if PSA group saw ads: {extra_conversions}")

# Overall extra conversions ads drove vs if everyone saw PSA
total_extra = int(ad_users * (ad_rate - psa_rate))
print(f"Extra conversions ads drove vs PSA baseline: {total_extra:,}")

Exploratory Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# PART 1 - Conversion rate by day of week
# ============================================

print(" CONVERSION RATE BY DAY ")
day_analysis = df.groupby('most ads day')['converted'].agg(['sum', 'count', 'mean'])
day_analysis.columns = ['Conversions', 'Total Users', 'Conversion Rate']
day_analysis['Conversion Rate %'] = (day_analysis['Conversion Rate'] * 100).round(2)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_analysis = day_analysis.reindex(day_order)
print(day_analysis)

# Plot
plt.figure(figsize=(10, 5))
sns.barplot(x=day_analysis.index, y=day_analysis['Conversion Rate %'], color='steelblue')
plt.title('Conversion Rate by Day of Week', fontsize=14)
plt.xlabel('Day')
plt.ylabel('Conversion Rate %')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ============================================
# PART 2 - Conversion rate by hour of day
# ============================================

print("\n CONVERSION RATE BY HOUR ")
hour_analysis = df.groupby('most ads hour')['converted'].agg(['sum', 'count', 'mean'])
hour_analysis.columns = ['Conversions', 'Total Users', 'Conversion Rate']
hour_analysis['Conversion Rate %'] = (hour_analysis['Conversion Rate'] * 100).round(2)
print(hour_analysis)

# Plot
plt.figure(figsize=(14, 5))
sns.lineplot(x=hour_analysis.index, y=hour_analysis['Conversion Rate %'], marker='o', color='steelblue')
plt.title('Conversion Rate by Hour of Day', fontsize=14)
plt.xlabel('Hour (0 = Midnight, 12 = Noon)')
plt.ylabel('Conversion Rate %')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

Ad Fatigue Analysis

In [ ]:
print(" AD FATIGUE ANALYSIS ")

# Look at conversion rate by number of ads seen
# First let's see the distribution of total ads
print("Total ads distribution (percentiles):")
print(df['total ads'].describe(percentiles=[.25, .50, .75, .90, .95, .99]))

# Create ad frequency buckets
df['ad_bucket'] = pd.cut(df['total ads'], 
                          bins=[0, 5, 10, 20, 50, 100, 200, 500, 2000],
                          labels=['1-5', '6-10', '11-20', '21-50', 
                                  '51-100', '101-200', '201-500', '500+'])

# Conversion rate by bucket - only for ad group
ad_group = df[df['test group'] == 'ad']
fatigue_analysis = ad_group.groupby('ad_bucket', observed=True)['converted'].agg(['sum', 'count', 'mean'])
fatigue_analysis.columns = ['Conversions', 'Total Users', 'Conversion Rate']
fatigue_analysis['Conversion Rate %'] = (fatigue_analysis['Conversion Rate'] * 100).round(2)
print("\nConversion Rate by Ad Frequency:")
print(fatigue_analysis)

# Plot
plt.figure(figsize=(12, 5))
sns.lineplot(x=fatigue_analysis.index, y=fatigue_analysis['Conversion Rate %'], 
             marker='o', color='steelblue', linewidth=2.5)
plt.title('Ad Fatigue Analysis — Conversion Rate by Number of Ads Seen', fontsize=14)
plt.xlabel('Number of Ads Seen')
plt.ylabel('Conversion Rate %')
plt.tight_layout()
plt.show()